In [ ]:
import os, sys, json, random, time
import numpy as np
import pandas as pd
import igraph as ig
from pathlib import Path

# One seed, set everywhere, so this notebook reproduces exactly.
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

# Paths are relative to notebooks/, so ROOT lands on the repo root.
ROOT = Path("..").resolve()
DATA_PROC = ROOT / "data/processed"
OUT_DIR = ROOT / "outputs"


print("igraph version:", ig.__version__)
print("Graph present:", (DATA_PROC / "04_graph_train.pkl").exists())

igraph version: 0.11.9
Graph present: True


In [3]:
df = pd.read_parquet(DATA_PROC / "01_audit.parquet")
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
print(f"Loaded {len(df):,} transactions")

# 70/15/15 cut points, taken from the timestamp itself.
t70 = df["Timestamp"].quantile(0.70)
t85 = df["Timestamp"].quantile(0.85)

# Each row gets its label from its own timestamp, so row order cannot corrupt it.
df["split"] = np.where(df["Timestamp"] <= t70, "train",
              np.where(df["Timestamp"] <= t85, "val", "test"))

# Stop the notebook if the rebuild disagrees with the registered split by even one row.
counts = df["split"].value_counts().to_dict()
assert counts == {"train": 3554957, "val": 761749, "test": 761639}, f"Split mismatch: {counts}"
print("Split reproduced:", counts)

Loaded 5,078,345 transactions
Split reproduced: {'train': 3554957, 'val': 761749, 'test': 761639}


In [4]:
t0 = time.time()
g = ig.Graph.Read_Pickle(str(DATA_PROC / "04_graph_train.pkl"))
print(f"Loaded graph in {time.time() - t0:.1f}s")

# The graph must be exactly the one registered, or every feature below is off-plan.
assert g.vcount() == 513284, f"Vertex count mismatch: {g.vcount():,}"
assert g.ecount() == 3554957, f"Edge count mismatch: {g.ecount():,}"
print(f"Vertices: {g.vcount():,}  Edges: {g.ecount():,}  Directed: {g.is_directed()}")

# Self-loops are accounts paying themselves; they complicate "distinct counterparty".
n_loops = sum(g.is_loop())
print(f"Self-loops: {n_loops:,}")

# How often does the same pair transact repeatedly? This is the gap between the two degree definitions.
mult = np.array(g.count_multiple())
print(f"Edges sitting on a repeated pair: {(mult > 1).sum():,} ({(mult > 1).mean():.1%})")
print(f"Distinct ordered account pairs: {len(set(g.get_edgelist())):,}")

Loaded graph in 1.7s
Vertices: 513,284  Edges: 3,554,957  Directed: True
Self-loops: 558,821
Edges sitting on a repeated pair: 3,068,302 (86.3%)
Distinct ordered account pairs: 955,904


In [5]:
SENDER_COL, RECEIVER_COL = "Account", "Account.1"

# Cast to str so the comparison matches how Notebook 04 built the vertices.
train = df[df["split"] == "train"].copy()
train[SENDER_COL] = train[SENDER_COL].astype(str)
train[RECEIVER_COL] = train[RECEIVER_COL].astype(str)

# A self-loop is simply a row whose sender and receiver are the same account.
loop_mask = train[SENDER_COL] == train[RECEIVER_COL]
print(f"Self-loop rows: {loop_mask.sum():,} of {len(train):,} ({loop_mask.mean():.1%})")
print(f"Matches graph self-loop count: {loop_mask.sum() == 558821}")
print(f"Distinct accounts doing it: {train.loc[loop_mask, SENDER_COL].nunique():,}\n")

# Compare the two groups on the features that separate a conversion from a transfer.
for name, sub in [("SELF-LOOP", train[loop_mask]), ("OTHER", train[~loop_mask])]:
    same_ccy = (sub["Payment Currency"] == sub["Receiving Currency"]).mean()
    same_amt = np.isclose(sub["Amount Paid"], sub["Amount Received"]).mean()
    print(f"{name}: n={len(sub):,} | illicit={sub['Is Laundering'].mean():.4%} "
          f"| same currency={same_ccy:.1%} | paid==received={same_amt:.1%}")

# Payment format is the clearest signal of what kind of movement this is.
print("\nPayment format share, self-loops vs other:")
fmt = pd.DataFrame({
    "self_loop_%": train.loc[loop_mask, "Payment Format"].value_counts(normalize=True) * 100,
    "other_%":     train.loc[~loop_mask, "Payment Format"].value_counts(normalize=True) * 100,
}).round(1)
print(fmt.fillna(0))

# Twenty actual rows, so we are reading data rather than summary statistics.
cols = ["Timestamp", SENDER_COL, "Amount Paid", "Payment Currency",
        "Amount Received", "Receiving Currency", "Payment Format", "Is Laundering"]
print("\nSample of 20 self-loop rows:")
print(train.loc[loop_mask, cols].sample(20, random_state=SEED).to_string(index=False))

Self-loop rows: 558,821 of 3,554,957 (15.7%)
Matches graph self-loop count: True
Distinct accounts doing it: 367,282

SELF-LOOP: n=558,821 | illicit=0.0011% | same currency=91.7% | paid==received=91.7%
OTHER: n=2,996,136 | illicit=0.0951% | same currency=100.0% | paid==received=100.0%

Payment format share, self-loops vs other:
                self_loop_%  other_%
Payment Format                      
ACH                     8.5     11.8
Bitcoin                 4.0      2.7
Cash                    0.2     10.9
Cheque                  0.7     41.4
Credit Card             0.5     29.4
Reinvestment           86.1      0.0
Wire                    0.1      3.9

Sample of 20 self-loop rows:
          Timestamp   Account  Amount Paid Payment Currency  Amount Received Receiving Currency Payment Format  Is Laundering
2022-09-01 00:24:00 810351BA0        54.38           Shekel            54.38             Shekel   Reinvestment              0
2022-09-01 00:11:00 8075D2520      1693.63             

In [6]:
# Work on the whole dataset here, not just train, so the claim covers the benchmark.
full = df.copy()
full[SENDER_COL] = full[SENDER_COL].astype(str)
full[RECEIVER_COL] = full[RECEIVER_COL].astype(str)

full["is_self_loop"] = full[SENDER_COL] == full[RECEIVER_COL]
full["cross_ccy"] = full["Payment Currency"] != full["Receiving Currency"]

# The core question: of all cross-currency rows, what share are an account converting with itself?
xt = pd.crosstab(full["cross_ccy"], full["is_self_loop"])
print("Rows by cross-currency x self-loop:\n", xt, "\n")

n_cross = full["cross_ccy"].sum()
n_cross_loop = (full["cross_ccy"] & full["is_self_loop"]).sum()
print(f"Cross-currency rows: {n_cross:,} ({full['cross_ccy'].mean():.2%} of dataset)")
print(f"...of which self-loops: {n_cross_loop:,} ({n_cross_loop / n_cross:.2%})")

# If cross-currency movement almost never happens between two parties, say so precisely.
n_cross_between = n_cross - n_cross_loop
print(f"...cross-currency between DIFFERENT accounts: {n_cross_between:,}\n")

# Does any of it carry laundering signal?
print(f"Illicit among cross-currency rows: {full.loc[full['cross_ccy'], 'Is Laundering'].sum():,}")
print(f"Illicit among cross-currency self-loops: "
      f"{full.loc[full['cross_ccy'] & full['is_self_loop'], 'Is Laundering'].sum():,}")

# Which currencies the conversions run between, top pairs.
print("\nTop currency pairs among cross-currency rows:")
print(full.loc[full["cross_ccy"]]
        .groupby(["Payment Currency", "Receiving Currency"]).size()
        .sort_values(ascending=False).head(8).to_string())

Rows by cross-currency x self-loop:
 is_self_loop    False   True 
cross_ccy                    
False         4484942  521233
True             2191   69979 

Cross-currency rows: 72,170 (1.42% of dataset)
...of which self-loops: 69,979 (96.96%)
...cross-currency between DIFFERENT accounts: 2,191

Illicit among cross-currency rows: 0
Illicit among cross-currency self-loops: 0

Top currency pairs among cross-currency rows:
Payment Currency  Receiving Currency
US Dollar         Euro                  15838
Euro              US Dollar             11060
Yuan              US Dollar              6675
US Dollar         Yuan                   2547
                  Swiss Franc            2507
                  UK Pound               2489
                  Rupee                  2310
                  Shekel                 2204
